# P1 · EEG · 01: Kiểm tra pipeline control và PRR bằng embedding ATM đã công bố

**Mục đích:** chạy thử toàn bộ đường đi của shuf-x, shuf-w, metric và PRR trên dữ liệu thật, trước khi train decoder của mình.

**Không dùng kết quả này để ra quyết định hay đưa vào paper.** Embedding EEG trong `emb_eeg/` do tác giả ATM xuất ra từ checkpoint của họ. Nhánh `main` của ATM chọn checkpoint theo độ chính xác trên test, nên số liệu có thể bị thổi phồng.

Vì encoder xử lý từng trial độc lập, xáo tín hiệu đầu vào rồi mã hoá cho cùng kết quả với mã hoá rồi xáo embedding. Do đó shuffle chạy được ở mức embedding mà không cần model. Null và PCA thì cần model, nên chưa có trong notebook này.

Cần chạy notebook 00 trước để có metadata, đặc trưng CLIP và bảng category.

## 1. Nối Google Drive và chọn thư mục

Mọi dữ liệu và kết quả lâu dài nằm trên Drive. Ổ `/content` của Colab bị xoá khi phiên kết thúc, nên chỉ dùng làm chỗ tạm.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/BrainDecoding')   # đổi nếu chị muốn để chỗ khác
DATA = DRIVE / 'data'
EEG_DIR = DATA / 'things_eeg2' / 'eeg_63ch_250hz_f32'  # float32: đúng kiểu mà code NICE và ATM đưa vào model
META_DIR = DATA / 'things_meta'
RESULTS = DRIVE / 'results'
LOGS = DRIVE / 'logs'
for p in [EEG_DIR, META_DIR, RESULTS, LOGS, DATA / 'things_eeg2' / 'archives']:
    p.mkdir(parents=True, exist_ok=True)
SCRATCH = Path('/content/scratch'); SCRATCH.mkdir(exist_ok=True)
print('Drive:', DRIVE)

## 2. Lấy code

Chọn **một** trong hai cách:

- **Cách A, khuyến nghị:** đẩy repo lên GitHub ở chế độ private rồi clone. Cần một token đọc repo, lưu trong mục Secrets của Colab với tên `GH_TOKEN`.
- **Cách B:** trên máy chị chạy `bash scripts/pack_for_colab.sh`, rồi tải file zip lên `MyDrive/BrainDecoding/code/`. File zip chỉ chứa code đã commit, không chứa dữ liệu.

In [ ]:
import os, subprocess, zipfile
REPO = Path('/content/brain_decoding')
GET_CODE = 'zip'   # 'git' hoặc 'zip'
GIT_URL = 'github.com/<tai-khoan>/<ten-repo>.git'   # chỉ dùng với 'git'

if REPO.exists():
    print('Code đã có ở', REPO)
elif GET_CODE == 'git':
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://{token}@{GIT_URL}', str(REPO)], check=True)
else:
    zips = sorted((DRIVE / 'code').glob('brain_decoding_*.zip'))
    assert zips, 'Chưa có file zip trong MyDrive/BrainDecoding/code/'
    zipfile.ZipFile(zips[-1]).extractall(REPO)
    print('Đã giải nén', zips[-1].name)
os.chdir(REPO)
import sys; sys.path.insert(0, str(REPO))

In [ ]:
# Cài các gói còn thiếu. Không cài lại torch: dùng bản CUDA sẵn có của Colab.
!pip install -q -r requirements-colab.txt
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# Ghi cấu hình máy và phiên bản gói lên Drive để tái lập được kết quả
import datetime, json, torch, platform, psutil
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
info = {
    'time': stamp, 'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'gpu': gpu,
    'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if gpu != 'none' else 0,
    'ram_gb': round(psutil.virtual_memory().total / 1e9, 1),
    'disk_free_gb_content': round(psutil.disk_usage('/content').free / 1e9, 1),
    'commit': subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'zip',
}
print(info)
(LOGS / f'colab_env_{stamp}.json').write_text(json.dumps(info, indent=1))
!pip freeze > {LOGS}/pip_freeze_{stamp}.txt

In [ ]:
import torch, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download
HF_REPO = 'LidongYang/EEG_Image_decode'
SUB = 'sub-01'
p_eeg = hf_hub_download(HF_REPO, f'emb_eeg/ATM_S_eeg_features_{SUB}_test.pt', repo_type='dataset', local_dir=SCRATCH / 'hf')
eeg = torch.load(p_eeg, map_location='cpu', weights_only=False)
img = torch.load(DATA / 'things_eeg2' / 'archives' / 'ViT-H-14_features_test.pt', map_location='cpu', weights_only=False)
print('emb EEG:', type(eeg), getattr(eeg, 'shape', None) if not isinstance(eeg, dict) else {k: getattr(v, 'shape', v) for k, v in eeg.items()})
print('emb ảnh:', type(img), {k: getattr(v, 'shape', None) for k, v in img.items()} if isinstance(img, dict) else img.shape)

In [ ]:
# Chuẩn hoá định dạng: cần pred (200, d) và target (200, d) cùng thứ tự ảnh test
pred = eeg if not isinstance(eeg, dict) else next(v for v in eeg.values() if hasattr(v, 'shape'))
target = img['img_features'] if isinstance(img, dict) else img
pred = pred.float().numpy().reshape(200, -1); target = target.float().numpy().reshape(200, -1)
assert pred.shape == target.shape, (pred.shape, target.shape)
cats = pd.read_csv(RESULTS / 'test_categories_manual.csv')
groups = cats['label_exclude_multi'].to_numpy()
ids = cats['concept'].tolist()
print(pred.shape, 'ảnh dùng được cho shuf-w:', (groups >= 0).sum())

In [ ]:
from brain.controls import shuffle_within_category, shuffle_cross_category, apply_donor
from brain.tier1 import Condition, evaluate, summarize_prr
N_PERM = 50
conds = [Condition('real', pred)]
for s in range(N_PERM):
    rng = np.random.default_rng(s)
    conds.append(Condition('shuf-x', pred[shuffle_cross_category(groups, rng)], seed=s))
    d = shuffle_within_category(groups, rng)
    x, keep = apply_donor(pred, d)
    conds.append(Condition('shuf-w', x, items=keep, seed=s))
df = evaluate(conds, target, groups, ids)
summary = summarize_prr(df, n_items=len(ids), n_boot=1000)
display(summary.round(3))
out = RESULTS / 'pipeline_check'; out.mkdir(exist_ok=True)
summary.to_csv(out / f'atm_published_{SUB}_prr_summary.csv', index=False)
print('Đã lưu. Nhắc lại: chỉ để kiểm tra pipeline, không dùng làm kết quả.')